# Introduction to LangChain
In this notebook, we will explore how LangChain organizes LLM pipelines using the official [LangChain Expression Language (LCEL)](https://www.langchain.com/blog/langchain-expression-language). We will build Prompt Templates, load chat models, and chain them seamlessly.

## Installation and Package Requirements
We install both the core LangChain framework package and the dedicated OpenAI integration module.

In [1]:
import os
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_ollama import ChatOllama
from langchain_core.output_parsers import StrOutputParser

## Dynamic Chat Prompt Templates
Instead of managing simple string concatenations, LangChain introduces structured prompt wrappers to pass clean system/user message arrays.

In [2]:
system_template = "Vous êtes un traducteur expert spécialisé dans la traduction de textes vers la langue suivante: {target_language}."
user_template = "Traduis ce texte pour moi: {input_text}"

prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_template),
    ("user", user_template)
])

# Preview the structured prompt data
preview = prompt_template.invoke({"target_language": "Spanish", "input_text": "Bonjour, bienvenue au séminaire de formation sur les agents IA"})
print(preview)

messages=[SystemMessage(content='Vous êtes un traducteur expert spécialisé dans la traduction de textes vers la langue suivante: Spanish.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Traduis ce texte pour moi: Bonjour, bienvenue au séminaire de formation sur les agents IA', additional_kwargs={}, response_metadata={})]


## Building an LCEL Chain (Prompt | Model | OutputParser)
Using LangChain's pipeline operator (`|`), we construct a linear chain. Inputs automatically feed into the prompt, the formatted prompt streams into OpenAI, and the parser extracts the raw string text response.

In [6]:
# Initialize modern ChatOpenAI object wrapper
# model = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
model = ChatOllama(model="mistral:7b", temperature=0.3)

output_parser = StrOutputParser()

In [7]:
translation_chain = prompt_template | model | output_parser


In [8]:


# Create the pipeline chain using the pipe operator

# Execute the chain invocation pipeline
result = translation_chain.invoke({
    "target_language": "English",
    "input_text": "Bonjour, bienvenue au séminaire de formation sur les agents IA"
})

print("--- Translation Result ---")
print(result)

--- Translation Result ---
 Hello, welcome to the seminar on Artificial Intelligence training.


## Going forward
Build a custom troubleshooting assistant pipeline chain. 
1. Define a `ChatPromptTemplate` that takes the code as input: `{error_code}`.
2. Chain your new prompt into the existing `model` and `output_parser` variables.
3. Invoke your chain using Python and a sample error message (e.g., KeyError or ValueError).

In [ ]:
# TODO: Build your chain below
# prompt = ChatPromptTemplate.from_messages(...)
# chain = prompt | model | output_parser
# print(chain.invoke({...}))


### Solution Hint
Uncomment and run below to check your answer if you get stuck.

In [12]:
exercise_prompt = ChatPromptTemplate.from_messages([
    ("system", "Tu es un ingénieur principal DevOps expert en résolution d'erreurs.\
Analyse l'erreur fournie par l'utilisateur et explique les raisons qui la justifient. \
Donne une explication concise (maximum 3 phrases) de la cause probable et propose la commande exacte ou la modification de code pour la corriger. \
Sois direct, technique, et précis.."),
    ("user", "Language: {programming_language}\nError: {error_code}")
])


In [13]:
debugging_chain = exercise_prompt | model | output_parser

In [17]:
output = debugging_chain.invoke(
    {"programming_language": "Python", 
     "error_code": "KeyError: 'user_id' not found"
    })


In [19]:
print(output)

L'erreur `KeyError: 'user_id' not found` indique que le code tente d'accéder à une clé `'user_id'` dans un dictionnaire ou un objet similaire, mais cette clé n'existe pas. Cela peut se produire si la clé n'a pas été définie ou si les données n'ont pas été correctement chargées. Pour corriger cela, assurez-vous que la clé existe avant d'y accéder, par exemple :

```python
user_id = data.get('user_id', default_value)  # Remplacez default_value par une valeur par défaut appropriée
```

Ou, si vous êtes certain que la clé doit exister, vous pouvez vérifier les données avant l'accès :

```python
if 'user_id' in data:
    user_id = data['user_id']
else:
    raise KeyError("La clé 'user_id' est manquante dans les données.")
```
